# V4 DINOv3 ViT-L SAT-493M profile

This Kaggle-only gate loads the pinned satellite ViT-L checkpoint and the matching DINOv3 repository from attached Kaggle sources. It runs one real xBD before/after pair at the native 512-pixel crop size with no gradients, validates spatial patch-token output, and writes a memory/throughput report. It does not train, tune thresholds, or access the held-out split.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_PATH = Path("/kaggle/working/rv4_dinov3_vitl_profile.json")
MODEL_NAME = "dinov3_vitl16_pretrain_sat493m"
MODEL_REVISION = "facebook/dinov3-vitl16-pretrain-sat493m"
PATCH_SIZE = 16
CROP_SIZE = 512


def first_path(pattern: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(pattern))
    if not matches:
        raise RuntimeError(f"required Kaggle input is missing: {pattern}")
    return matches[0]


def load_rgb(path: Path) -> torch.Tensor:
    with Image.open(path) as image:
        array = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    array = array[:CROP_SIZE, :CROP_SIZE, :]
    if array.shape != (CROP_SIZE, CROP_SIZE, 3):
        raise RuntimeError(f"unexpected crop shape for {path}: {array.shape}")
    return torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)


def extract_patch_tokens(output):
    if isinstance(output, dict):
        for key in ("x_norm_patchtokens", "patch_tokens", "last_hidden_state"):
            value = output.get(key)
            if torch.is_tensor(value):
                return value, key
    for key in ("x_norm_patchtokens", "patch_tokens", "last_hidden_state"):
        value = getattr(output, key, None)
        if torch.is_tensor(value):
            return value, key
    if torch.is_tensor(output):
        return output, "tensor"
    raise RuntimeError(f"unsupported DINOv3 feature output type: {type(output).__name__}")


def main() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("ViT-L profile requires CUDA; refusing a CPU fallback")
    device = torch.device("cuda")
    hubconf_paths = sorted(INPUT_ROOT.rglob("hubconf.py"))
    if not hubconf_paths:
        raise RuntimeError("the attached DINOv3 repository has no hubconf.py")
    selected_hubconf = None
    discovered = []
    for hubconf in hubconf_paths:
        spec = importlib.util.spec_from_file_location("rv4_dinov3_hubconf", hubconf)
        if spec is None or spec.loader is None:
            continue
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        names = sorted(name for name in dir(module) if name.startswith("dinov3") and callable(getattr(module, name)))
        discovered.append({"path": str(hubconf), "callables": names})
        if MODEL_NAME in names:
            selected_hubconf = hubconf
            break
    print(json.dumps({"hubconf_candidates": discovered}, indent=2, sort_keys=True))
    if selected_hubconf is None:
        raise RuntimeError(f"DINOv3 model entry point {MODEL_NAME!r} is absent from attached hubconf files")
    repo_root = selected_hubconf.parent
    weights = first_path("*.pth")
    pre_path = first_path("*_pre_disaster.png")
    post_path = pre_path.with_name(pre_path.name.replace("_pre_disaster.png", "_post_disaster.png"))
    if not post_path.exists():
        raise RuntimeError(f"paired post-disaster image is missing: {post_path}")
    sys.path.insert(0, str(repo_root))
    torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    model = torch.hub.load(str(repo_root), MODEL_NAME, source="local", weights=str(weights))
    model = model.to(device).eval()
    load_seconds = time.perf_counter() - started
    pre = load_rgb(pre_path).to(device)
    post = load_rgb(post_path).to(device)
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
        torch.cuda.synchronize(device)
        infer_started = time.perf_counter()
        pre_output = model.forward_features(pre) if hasattr(model, "forward_features") else model(pre)
        post_output = model.forward_features(post) if hasattr(model, "forward_features") else model(post)
        torch.cuda.synchronize(device)
        infer_seconds = time.perf_counter() - infer_started
    pre_tokens, pre_key = extract_patch_tokens(pre_output)
    post_tokens, post_key = extract_patch_tokens(post_output)
    expected_tokens = (CROP_SIZE // PATCH_SIZE) ** 2
    if pre_tokens.ndim != 3 or post_tokens.ndim != 3:
        raise RuntimeError(f"patch tokens must be [B,N,C], got {tuple(pre_tokens.shape)} and {tuple(post_tokens.shape)}")
    if pre_tokens.shape != post_tokens.shape:
        raise RuntimeError(f"pre/post token shapes differ: {tuple(pre_tokens.shape)} vs {tuple(post_tokens.shape)}")
    if pre_tokens.shape[1] != expected_tokens:
        raise RuntimeError(f"expected {expected_tokens} spatial tokens for {CROP_SIZE}px/{PATCH_SIZE}px, got {pre_tokens.shape[1]}")
    report = {
        "schema_version": "rv4-dinov3-vitl-profile-1",
        "status": "PASS",
        "model_revision": MODEL_REVISION,
        "model_name": MODEL_NAME,
        "weights": str(weights),
        "repository": str(repo_root),
        "device": str(device),
        "gpu_name": torch.cuda.get_device_name(device),
        "torch_version": torch.__version__,
        "pair": {"pre": str(pre_path), "post": str(post_path)},
        "input_shape": list(pre.shape),
        "pre_output_key": pre_key,
        "post_output_key": post_key,
        "token_shape": list(pre_tokens.shape),
        "expected_spatial_tokens": expected_tokens,
        "embedding_dim": int(pre_tokens.shape[-1]),
        "model_parameter_count": sum(parameter.numel() for parameter in model.parameters()),
        "load_seconds": load_seconds,
        "paired_inference_seconds": infer_seconds,
        "peak_memory_allocated": int(torch.cuda.max_memory_allocated(device)),
        "peak_memory_reserved": int(torch.cuda.max_memory_reserved(device)),
    }
    OUTPUT_PATH.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print(json.dumps(report, indent=2, sort_keys=True))


if __name__ == "__main__":
    main()
